# Topic Hierarchy — iGEM Teams (Low → Mid → High)

Builds a three-level hierarchy for the existing **iGEM Teams** topic model
by cutting BERTopic's agglomerative merge tree at two levels:

- **mid** — auto-selected by silhouette over `[HIGH_K_MAX + 1, n_low // 3]`
- **high** — auto-selected by silhouette over `[HIGH_K_MIN, HIGH_K_MAX]`

**Inputs:** `teams_topic_model`, `teams_doc_topics.txt`, `teams_topic_names.txt`, `teams_corpus.txt`, `igem.txt`

**Outputs (in `assets/reports/`):**
- `teams_topic_hierarchy_map.tsv` — document-level mapping (`UT, low, mid, high`)
- `teams_topic_name_hierarchy.tsv` — low-level names mapped to `low, mid, high`
- `teams_topic_hierarchy_summary.tsv` — mid/high group summary stats

In [1]:
# ── Make the shared aux/ package importable ───────────────────────────────────
# These notebooks live one level below 04-topic_hierarchy/, where the aux/
# package resides; add that folder to the import path.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd

from aux.paths import REPORTS_DIR, HIGH_K_MIN, HIGH_K_MAX, set_seed
from aux.hierarchy import load_hierarchy_inputs, select_hierarchy_levels, write_hierarchy_reports

set_seed()

# ── CONFIG: iGEM Teams ──────────────────────────────────────────────────
PREFIX = "teams"
ID_COL = "UT"
YEAR_COL = "Year_y"
RAW_FILE = "igem.txt"
RENAME_ID_FROM = None   # source id column to rename to ID_COL (None if already named)

model, doc_topics, topic_names, corpus, raw = load_hierarchy_inputs(
    PREFIX, id_col=ID_COL, year_col=YEAR_COL, raw_filename=RAW_FILE, rename_id_from=RENAME_ID_FROM,
)
print(f"{PREFIX}: {len(doc_topics):,} docs, {len(topic_names):,} topics, "
      f"outliers={(doc_topics['low'] == -1).sum():,}")

teams: 3,811 docs, 154 topics, outliers=0


## 1. Build hierarchy and auto-select mid / high levels

In [3]:
corpus_texts = corpus["text"].astype(str).tolist()
hierarchy_map, sel = select_hierarchy_levels(model, corpus_texts, HIGH_K_MIN, HIGH_K_MAX)

print(f"\nHigh K = {sel['high_k']} (silhouette {sel['high_score']:.4f})  |  "
      f"Mid K = {sel['mid_k']} (silhouette {sel['mid_score']:.4f})")
print("\n--- High-level candidates ---")
display(pd.DataFrame(sel["high_scores"], columns=["high_k", "silhouette"]).sort_values("high_k"))
print("--- Mid-level candidates ---")
pd.DataFrame(sel["mid_scores"], columns=["mid_k", "silhouette"]).sort_values("mid_k")

  Building BERTopic hierarchical merge tree …


100%|██████████| 153/153 [00:01<00:00, 127.61it/s]


  Non-outlier low topics: 154
  Tracking cluster maps for k = 1 … 154
  Captured 153 distinct k-level snapshots
  Scoring high-level k candidates in [4, 11] …
    k=  4  silhouette=0.0707
    k=  5  silhouette=0.0659
    k=  6  silhouette=0.0684
    k=  7  silhouette=0.0475
    k=  8  silhouette=0.0581
    k=  9  silhouette=0.0668
    k= 10  silhouette=0.0781
    k= 11  silhouette=0.0438
  ✓ Selected high-level k = 10 (silhouette = 0.0781)
  Scoring mid-level k candidates in [12, 51] …
    k= 12  silhouette=0.0501
    k= 13  silhouette=0.0634
    k= 14  silhouette=0.0687
    k= 15  silhouette=0.0680
    k= 16  silhouette=0.0713
    k= 17  silhouette=0.0829
    k= 18  silhouette=0.0833
    k= 19  silhouette=0.0700
    k= 20  silhouette=0.0773
    k= 21  silhouette=0.0860
    k= 22  silhouette=0.0865
    k= 23  silhouette=0.0813
    k= 24  silhouette=0.0824
    k= 25  silhouette=0.0796
    k= 26  silhouette=0.0819
    k= 27  silhouette=0.0825
    k= 28  silhouette=0.0821
    k= 29  silho

,high_k,silhouette
0,4,0.070723
1,5,0.065905
2,6,0.068371
3,7,0.047549
4,8,0.058116
5,9,0.066829
6,10,0.078085
7,11,0.043770


--- Mid-level candidates ---


,mid_k,silhouette
0,12,0.050131
1,13,0.063411
2,14,0.068686
3,15,0.067969
4,16,0.071302
5,17,0.082921
6,18,0.083301
7,19,0.070049
8,20,0.077303
9,21,0.085994


## 2. Build and save the report tables

In [4]:
doc_map, name_map, summary = write_hierarchy_reports(
    doc_topics, topic_names, raw, hierarchy_map,
    id_col=ID_COL, year_col=YEAR_COL, prefix=PREFIX,
)
print(f"Saved 3 hierarchy reports for {PREFIX} → {REPORTS_DIR}")
name_map.head()

  Document map: 3,811 rows (0 outliers)
  Name map: 154 topics
  Summary: 41 rows (mid + high)
Saved 3 hierarchy reports for teams → /Users/cristian/Desktop/GitHub/igem-synbio/assets/reports


,global_name,low,mid,high
0,Gene Expression Patterning,0,0,0
1,Pest and Vector Genetic Control,1,1,1
2,Plant Disease Synthetic Diagnostics,2,1,1
3,Biofilm and Quorum Disruption,3,2,0
4,Synthetic Biology Design Automation,4,3,0


## 3. Validation

In [5]:
assert list(doc_map.columns) == [ID_COL, "low", "mid", "high"]
assert list(name_map.columns) == ["global_name", "low", "mid", "high"]
assert {"level", "group_id", "total_count", "avg_publication_year", "median_publication_year"}.issubset(summary.columns)
assert len(doc_map) == len(doc_topics)
assert (doc_map.loc[doc_map["low"] == -1, ["mid", "high"]] == -1).all().all()
assert HIGH_K_MIN <= sel["high_k"] <= HIGH_K_MAX
assert sel["mid_min"] <= sel["mid_k"] <= sel["mid_max"]
print("All validation checks passed ✓")

All validation checks passed ✓


## 4. Hierarchy quality summary

In [6]:
outlier_pct = (doc_topics["low"] == -1).mean() * 100
print(f"Selected high-level K : {sel['high_k']}  (silhouette = {sel['high_score']:.4f})")
print(f"Selected mid-level K  : {sel['mid_k']}  (silhouette = {sel['mid_score']:.4f})")
print(f"Outlier documents     : {outlier_pct:.2f}%")
print(f"Mid-level search range: [{sel['mid_min']}, {sel['mid_max']}]")

Selected high-level K : 10  (silhouette = 0.0781)
Selected mid-level K  : 31  (silhouette = 0.0880)
Outlier documents     : 0.00%
Mid-level search range: [12, 51]
